# Train Ultralytics YOLO from Scratch (Local)

Local version of `colab_notebooks/YOLO_Training_v2.ipynb`.

Trains a YOLO detector from the custom architecture config (`model_configurations/yolo_nivetha_v1.yaml`) with randomly initialized weights — no pretrained `.pt` weights are loaded.

- **Dataset:** `data/wildlife_dataset` (images/labels with train/val/test splits)
- **Model config:** `ai-applications/model_configurations/yolo_nivetha_v1.yaml`
- **Final trained model:** `ai-applications/trained_model`

## Dataset format

```text
wildlife_dataset/
├── images/train
├── images/val
├── images/test
├── labels/train
├── labels/val
└── labels/test
```

Each label file uses:

```text
class_id center_x center_y width height
```

All coordinates must be normalized from 0 to 1.

In [ ]:
%pip install -q -U ultralytics pyyaml

In [ ]:
import os
import shutil
import yaml
import torch
from pathlib import Path
from ultralytics import YOLO

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
PROJECT_ROOT = Path(r'C:\Users\nivet\ISY5002-PRS-G14')

DATASET_DIR = PROJECT_ROOT / 'data' / 'wildlife_dataset'
MODEL_CFG = PROJECT_ROOT / 'ai-applications' / 'model_configurations' / 'yolo_nivetha_v1.yaml'
TRAINED_MODEL_DIR = PROJECT_ROOT / 'ai-applications' / 'trained_model'
RUNS_DIR = PROJECT_ROOT / 'ai-applications' / 'yolo_runs'
RUN_NAME = 'wildlife_from_scratch'

assert DATASET_DIR.exists(), f'Dataset not found: {DATASET_DIR}'
assert MODEL_CFG.exists(), f'Model config not found: {MODEL_CFG}'
TRAINED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset:', DATASET_DIR)
print('Model config:', MODEL_CFG)
print('Trained model dir:', TRAINED_MODEL_DIR)

In [ ]:
# Generate the dataset YAML that Ultralytics expects
DATA = DATASET_DIR / 'wildlife_dataset.yaml'

dataset_config = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'buffalo', 1: 'elephant', 2: 'lion', 3: 'zebra'}
}

with open(DATA, 'w') as f:
    yaml.safe_dump(dataset_config, f, sort_keys=False)

print('Dataset YAML written to:', DATA)

In [ ]:
with open(DATA, 'r') as f:
    data = yaml.safe_load(f)

dataset_root = Path(data.get('path', DATA.parent))
if not dataset_root.is_absolute():
    dataset_root = DATA.parent / dataset_root

train_dir = dataset_root / data['train']
val_dir = dataset_root / data['val']
test_dir = dataset_root / data['test']
names = data['names']

print('Dataset root:', dataset_root)
print('Training images:', len([p for p in train_dir.rglob('*') if p.is_file()]))
print('Validation images:', len([p for p in val_dir.rglob('*') if p.is_file()]))
print('Test images:', len([p for p in test_dir.rglob('*') if p.is_file()]))
print('Classes:', names)

In [ ]:
model = YOLO(str(MODEL_CFG))
print('Model initialized from architecture YAML:', MODEL_CFG.name)
print('No pretrained .pt weights were loaded.')

In [ ]:
device = 0 if torch.cuda.is_available() else 'cpu'

results = model.train(
    data=str(DATA),
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    workers=2,
    pretrained=False,
    patience=25,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    plots=True,
    save=True,
    exist_ok=True
)

In [ ]:
best_path = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(str(best_path))

metrics = best_model.val(
    data=str(DATA),
    imgsz=640,
    device=device,
    plots=True
)

print('Best weights:', best_path)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## Save the final trained model

Copies the best checkpoint to `ai-applications/trained_model`.

In [ ]:
final_model_path = TRAINED_MODEL_DIR / 'yolo_nivetha_v1_best.pt'
shutil.copy2(best_path, final_model_path)

last_path = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
if last_path.exists():
    shutil.copy2(last_path, TRAINED_MODEL_DIR / 'yolo_nivetha_v1_last.pt')

print('Final trained model saved to:', final_model_path)

## Evaluate on the test split

In [ ]:
test_metrics = best_model.val(
    data=str(DATA),
    split='test',
    imgsz=640,
    device=device,
    plots=True
)

print('Test mAP50:', test_metrics.box.map50)
print('Test mAP50-95:', test_metrics.box.map)

## Inference on a sample test image

In [ ]:
TEST_IMAGE = next(p for p in test_dir.iterdir() if p.is_file())
print('Test image:', TEST_IMAGE)

prediction = best_model.predict(
    source=str(TEST_IMAGE),
    conf=0.35,
    imgsz=640,
    device=device,
    save=True
)

prediction[0].show()

In [ ]:
for result in prediction:
    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print({
            'class': result.names[class_id],
            'confidence': round(confidence, 4),
            'box': [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)]
        })

## Optional: export to ONNX

In [ ]:
onnx_path = best_model.export(
    format='onnx',
    imgsz=640
)

final_onnx_path = TRAINED_MODEL_DIR / 'yolo_nivetha_v1_best.onnx'
shutil.copy2(onnx_path, final_onnx_path)
print('ONNX model saved to:', final_onnx_path)